<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.9: Debugging and Verification
**Prev: [A Bus and a UART](5.8_bus_and_mmio.ipynb)**<br>
**Next: [From Chisel to an FPGA](5.10_to_silicon.ipynb)**<br>

## Motivation
Every core of this module has been checked the same way: run a program on the core and on the
golden model, and compare what they retire. This notebook looks at that method, and at two more,
on a bug planted in the forwarding unit of [Module 5.7](5.7_hazards.ipynb):

- the **retire trace**, which says *which* instruction went wrong;
- **waveforms**, which say *why*, signal by signal and cycle by cycle;
- a **formal check**, which proves a unit right for *every* input, or finds an input that breaks
  it, without a test program.

The cells that build a core stop with a `NotImplementedError` until the exercises of 5.2-5.7 are
done (the forwarding core uses them all, `Forward` of 5.7 included).

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._
import riscv._

// Formal.prove (below), through load-ivy.sc's scrub: Almond cannot show an error raised while Chisel
// builds a module (such as an unfinished exercise) without it
def prove(check: => RawModule): Option[String] = scrub(Formal.prove(check))

---
# The retire trace

Each core has a **retire port** (`io.retire`): in each cycle in which an instruction completes, it
reports the instruction and everything it did. It follows RVFI, the RISC-V Formal Interface that
riscv-formal and many verification tools use, in a smaller form:

| `RetireInfo` | RVFI | Meaning |
|---|---|---|
| `valid` (of the `Valid` around `RetireInfo`) | `rvfi_valid` | an instruction retires in this cycle |
| `pc`, `inst` | `rvfi_pc_rdata`, `rvfi_insn` | its address and its word |
| `nextPc` | `rvfi_pc_wdata` | the address of the next instruction |
| `rd`, `rdValue` | `rvfi_rd_addr`, `rvfi_rd_wdata` | the register it writes (0: none) and the value |
| `mem` | `rvfi_mem_addr`, `rvfi_mem_rdata`/`wdata`, masks | its load or store |
| `halt` | `rvfi_halt`; `rvfi_trap` for `Illegal`, `Misaligned`, `BadAddress` | it stops the core, and why |

`CoSim.run` collects the retired instructions, and each prints as a line of a trace, disassembled
(`Rv32i.disassemble`). The golden model reports the same records, so the two traces can be
compared line by line: that is all `CoSim.compare` does.

In [ ]:
val fib = Asm(Programs.source("programs/src", "fib"))
simulate(new ForwardingSystem) { c =>
  CoSim.load(c, fib)
  CoSim.run(c).take(12).foreach(println)
}

---
# A planted bug

`OldestFirst` is a forwarding unit with a plausible mistake: when the instructions in both EX/MEM
and MEM/WB write the register, it forwards the one in MEM/WB, the *older* result. `ForwardingCore`
takes the unit to build as a parameter, so the bug goes into an otherwise correct core:

In [ ]:
class OldestFirst extends Forward {
  def writes(r: Valid[Result]) = r.valid && r.bits.rd =/= 0.U && r.bits.rd === io.rs
  io.value := MuxCase(
    io.regValue,
    Seq(writes(io.memWb) -> io.memWb.bits.value, writes(io.exMem) -> io.exMem.bits.value)
  )
}
def buggy = new CoreSystem(new ForwardingCore(forwardUnit = new OldestFirst))
val twice = "li a0, 1\nli a0, 2\nadd a1, a0, a0\necall" // the smallest program that shows it

Would the tests find it? `Programs.corpus` is every program of the course (but `hello`, which
needs the UART of 5.8), random programs (20 here) and the official tests; each is compared with the golden model:

In [ ]:
val corpus = Programs.corpus(seeds = 1L to 20L)
simulate(buggy) { c =>
  val found = corpus.filter { case (_, image) => CoSim.compare(c, image).isDefined }.map(_._1)
  println(s"${found.size} of ${corpus.size} programs find the bug: ${found.mkString(", ")}")
}

A program finds the bug only if it writes a register in two instructions in a row and reads it in
the next: most of the official tests do (they compute a value, then check it), `fact` does, some of the
random programs do, and `fib` and `sum` do not. A test suite finds a bug only if some
program happens to exercise it, which is why programs written by others, for other reasons, and
random ones are worth running.

`twice`, defined above, is the smallest program that shows it:

In [ ]:
simulate(buggy) { c =>
  println(CoSim.compare(c, Asm(twice)).getOrElse("no difference"))
  CoSim.load(c, Asm(twice))
  println("\nthe core's trace:")
  CoSim.run(c).foreach(println)
}
println("\nthe golden model's trace:")
new IsaSim(Asm(twice)).run().foreach(println)

`instruction 2` counts from 0: it is the third. The trace says which instruction is wrong (`add a1, a0, a0` wrote 2, not 4), and what it computed:
1 + 1, the first `li`'s value. It does not say why. For that, look inside.

---
# Waveforms

`simulate(..., waves = "name")` dumps every signal of the design in every cycle into a VCD file
(Module 2.6), which a waveform viewer (GTKWave, Surfer) shows. `riscv.Vcd` reads one in Scala, so
this notebook can print the signals that matter as a table: one row per cycle, read just before
the clock edge (what the registers take at that edge).

The units' instances are `rs1Data_f` and `rs2Data_f` (one per source register), and the pipeline
registers keep their names (`idEx_bits_inst` is the instruction in EX). `vcd(name)` gives a
signal as a function of time. The table samples each cycle just before its clock edge, from the
first instruction in EX to the `ecall` (the instructions fetched behind it are flushed), and shows
what the unit for `rs1` gets and forwards:

In [ ]:
simulate(buggy, waves = "oldest-first") { c =>
  CoSim.load(c, Asm(twice))
  CoSim.run(c)
}
val vcd = Vcd.read(wavesFile("oldest-first").get)

def signal(name: String)  = vcd(s"core.$name")
def unit(port: String)    = signal(s"rs1Data_f.io_$port") // the forwarding unit of rs1
def result(stage: String) = {                             // what the unit gets from a stage ahead: "x10 = 2", or dashes
  val (valid, rd, value) = (unit(s"${stage}_valid"), unit(s"${stage}_bits_rd"), unit(s"${stage}_bits_value"))
  (t: Long) => if (valid(t) == 1) f"x${rd(t)}%-2s = ${value(t)}%-6s" else "-" * 13
}
val (inEx, inst, pc) = (signal("idEx_valid"), signal("idEx_bits_inst"), signal("idEx_bits_pc"))
val (rs, forwarded)  = (unit("rs"), unit("value"))
val (exMem, memWb)   = (result("exMem"), result("memWb"))
def isEcall(t: Long) = Rv32i.decode(inst(t).toLong).exists(_.name == "ecall")
val cycles           = vcd.risingEdges("core.clock").map(_ - 1).zipWithIndex.filter { case (t, _) => inEx(t) == 1 }
assert(cycles.exists(c => isEcall(c._1)), "the ecall never reached EX")
println("cycle  in EX                     rs   EX/MEM rd = value   MEM/WB rd = value   forwarded")
for ((t, cycle) <- cycles.take(cycles.indexWhere(c => isEcall(c._1)) + 1)) {
  val text = Rv32i.disassemble(inst(t).toLong, pc(t).toLong)
  println(f"$cycle%5d  $text%-24s  x${rs(t)}%-2s ${exMem(t)}%19s ${memWb(t)}%19s   ${forwarded(t)}")
}

In the cycle in which `add a1, a0, a0` is in EX, both stages ahead hold a result for `x10` (`a0`):
2 from the second `li`, one stage ahead, and 1 from the first. The unit forwards 1. With a bug in
hand, a waveform is the place to confirm the cause; it is a poor place to search for one, as a
long program has thousands of cycles and hundreds of signals. The trace narrows the search to one
instruction, and so to a few cycles.

---
# A formal check

Co-simulation tries programs; a **formal** tool reasons about all inputs at once. `ForwardCheck`
puts a forwarding unit next to its specification (the newest valid result for `rs`, never for
`x0`, else the register file's value), written another way: with `when`s, the older result
connected first and the newer one after it, so that Chisel's last-connect rule makes the newer one
win. It has an output `ok` that is 1 when the two agree:

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "ForwardCheck.scala"))

`Formal.prove` emits the SystemVerilog of the check, and asks **Yosys** to prove that `ok` is 1
for every value of the inputs, with Yosys's own SAT solver (the `sat -prove` command). It takes a
combinational module: the check has no registers. The image
installs YoWASP's Yosys, compiled to WebAssembly; its first run in a container compiles it, which
takes about a minute.

`prove` (defined in the Setup) returns None when the proof succeeds, or the counterexample: input
values for which `ok` is 0. `check` prints either:

In [ ]:
def check(name: String, unit: => Forward): Unit =
  if (!Yosys.available) println(s"$name: no formal check (Yosys is not installed)")
  else
    println(
      prove(new ForwardCheck(unit)).fold(s"$name: proved for every input")(cex => s"$name, a counterexample:\n$cex")
    )

In [ ]:
check("Forward", new Forward)
check("OldestFirst", new OldestFirst)

The solver found the situation on its own: both stages valid and writing `rs`, with different
values. No program was needed, and the check covers all 2^113 combinations of its 113 input bits.

What it does not cover: the proof is about one unit against a specification someone wrote, so a
mistake in both is not found, nor is a bug in how the core wires the unit. And `sat -prove` checks
combinational logic: a property over several cycles (say, that a request is answered within 10 cycles) needs
a bounded model check over a number of steps, or induction, for which SymbiYosys is the usual
front end; it is not in the image. Co-simulation checks the whole core, but only for the programs
it runs. The two complement each other.

---
# Exercise: find the second bug

`ForwardsX0` makes another mistake: it forgets that `x0` is always 0, and forwards a result
"written" to it. Write a program (as assembly in `program`) that the correct core runs as the golden
model does, and on which the core with `ForwardsX0` differs from it. Then ask the formal check.

In [ ]:
class ForwardsX0 extends Forward {
  def writes(r: Valid[Result]) = r.valid && r.bits.rd === io.rs // no check for x0
  io.value := MuxCase(
    io.regValue,
    Seq(writes(io.exMem) -> io.exMem.bits.value, writes(io.memWb) -> io.memWb.bits.value)
  )
}

In [ ]:
def program: String = ??? // the assembly of your program, ending with ecall

In [ ]:
val image = Asm(program)
simulate(new ForwardingSystem) { c => assert(CoSim.compare(c, image).isEmpty, "the correct core should run it") }
simulate(new CoreSystem(new ForwardingCore(forwardUnit = new ForwardsX0))) { c =>
  val difference = CoSim.compare(c, image)
  assert(difference.isDefined, "the core with ForwardsX0 runs it as the model does: try again")
  println(difference.get)
}
check("ForwardsX0", new ForwardsX0)
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
def program: String = "addi zero, zero, 7\nadd a0, zero, zero\necall"

// addi writes 7 "to" x0 (and changes nothing); the next instruction reads x0 while that addi is
// one stage ahead, so ForwardsX0 forwards 7: a0 = 14 instead of 0. The formal check's
// counterexample has rs = 0.
</pre></article></div></section></div>

---
# You're done!

[Return to the top.](#top)